# 01 — Data Understanding

## Project: Customer Retention & Revenue Intelligence

In this notebook, we will explore the Online Retail II dataset using basic Python and Pandas. The goal is to understand what the data contains before cleaning it or building a machine-learning model.

**Learning rule:** Run one code cell at a time. Read the explanation, inspect the output, and write down what you notice. We will not use complicated functions or production code here.

### Dataset
The dataset contains retail transactions. Each row generally represents an item on an invoice. It includes invoice details, products, quantities, prices, customers and countries.

Expected file: `data/raw/online_retail_II.xlsx` (download it from the UCI Online Retail II dataset page and place it in this folder structure).


## 1. Import the libraries

Pandas helps us load and analyse tabular data. Matplotlib helps us create simple charts. `Path` helps us work with file paths.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


## 2. Find and load the Excel workbook

This notebook expects Jupyter to be opened from the `customer-retention-revenue-intelligence` project folder. The workbook contains multiple sheets, so we load the sheets and combine them into one DataFrame.

In [ ]:
data_path = Path("data/raw/online_retail_II.xlsx")

print("Workbook exists:", data_path.exists())
print("Workbook path:", data_path.resolve())


If `Workbook exists` is `False`, stop here. Download the dataset and save it at the path shown above. Do not continue until the file is available.

In [ ]:
# Read every worksheet into a dictionary of DataFrames
sheets = pd.read_excel(data_path, sheet_name=None)

print("Worksheet names:", list(sheets.keys()))

# Combine the worksheets into one DataFrame
df = pd.concat(sheets.values(), ignore_index=True)

print("Dataset loaded successfully!")


## 3. First look at the data

`head()` shows the first five rows. `shape` tells us the number of rows and columns. `columns` lists the column names.

In [ ]:
df.head()


In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("Column names:")
print(df.columns.tolist())


## 4. Understand column data types

`info()` shows each column's data type and how many non-missing values it contains. `describe()` gives summary statistics for numeric columns.

In [ ]:
df.info()


In [ ]:
df.describe()


## 5. Check missing values and duplicate rows

A missing value means a field has no recorded value. A duplicate row is a row repeated exactly. At this stage, we are only measuring them—we are not deleting anything.

In [ ]:
missing_values = df.isnull().sum().sort_values(ascending=False)
display(missing_values)

print("Exact duplicate rows:", df.duplicated().sum())


## 6. Understand the date range

The invoice date tells us when each transaction happened. Convert it to a datetime value so Pandas can work with dates.

In [ ]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")

print("Earliest invoice date:", df["InvoiceDate"].min())
print("Latest invoice date:", df["InvoiceDate"].max())


## 7. Explore quantities and prices

Look at the minimum, maximum, average and quartiles. Negative quantities can occur in returned or cancelled transactions, so do not remove them yet.

In [ ]:
display(df[["Quantity", "Price"]].describe())


## 8. Explore sales by country

First calculate line revenue as `Quantity × Price`. This is a simple exploratory measure; we will define the final valid-sales rules during data cleaning.

In [ ]:
df["LineRevenue"] = df["Quantity"] * df["Price"]

revenue_by_country = (
    df.groupby("Country")["LineRevenue"]
      .sum()
      .sort_values(ascending=False)
      .head(10)
)

display(revenue_by_country)


In [ ]:
revenue_by_country.sort_values().plot(kind="barh", figsize=(9, 5))
plt.title("Top 10 Countries by Recorded Line Revenue")
plt.xlabel("Line revenue")
plt.ylabel("Country")
plt.tight_layout()
plt.show()


## 9. Explore monthly transaction activity

Count the number of invoice rows per month. This is a first look at transaction activity, not yet a cleaned monthly sales metric.

In [ ]:
monthly_rows = df.set_index("InvoiceDate").resample("MS").size()

monthly_rows.plot(figsize=(11, 4))
plt.title("Number of Transaction Rows per Month")
plt.xlabel("Month")
plt.ylabel("Number of rows")
plt.tight_layout()
plt.show()


## 10. Your observations

Answer these in your own words. It is fine if you are unsure—we will discuss your answers together.

1. How many rows and columns are in the dataset?
2. Which columns contain the most missing values? Why might Customer ID be missing for some transactions?
3. What date range does the dataset cover?
4. Did you notice negative quantities or prices? What might they represent?
5. Which country has the highest recorded line revenue in this initial exploration?
6. What patterns do you notice in monthly transaction activity?

### Important
These are initial observations from raw data. We have not yet removed cancellations, handled missing customer IDs, removed duplicates, or defined valid purchases. We will make and explain those decisions in the next stage.
